# Kafka Project: Real-Time Analytics Data Platform


## Problem Statement

You are a Data Engineer building a real-time analytics system.

Requirements:
- Ingest real-time user events
- Process events instantly
- Store processed data
- Generate analytics

Goal:
👉 Build a production-style streaming pipeline


## System Architecture

Producer -> Kafka -> Consumer -> Transform -> Database -> Analytics

Optional:
Kafka -> Spark Streaming -> Data Lake


## Producer (Enhanced)

```python
from kafka import KafkaProducer
import json
import time
import random
from datetime import datetime

producer = KafkaProducer(
    bootstrap_servers='localhost:9092',
    value_serializer=lambda v: json.dumps(v).encode('utf-8')
)

events = ["click", "purchase", "login"]

while True:
    data = {
        "user_id": random.randint(1, 100),
        "event": random.choice(events),
        "amount": random.randint(100, 1000),
        "timestamp": datetime.now().isoformat()
    }
    producer.send("events-topic", value=data)
    print("Sent:", data)
    time.sleep(1)
```


## Consumer Pipeline

```python
from kafka import KafkaConsumer
import psycopg2
import json

consumer = KafkaConsumer(
    "events-topic",
    bootstrap_servers='localhost:9092',
    value_deserializer=lambda x: json.loads(x.decode('utf-8')),
    auto_offset_reset='earliest'
)

conn = psycopg2.connect(
    host="localhost",
    database="de_course",
    user="admin",
    password="admin"
)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS events (
    user_id INT,
    event TEXT,
    amount INT,
    timestamp TIMESTAMP
)
""")
conn.commit()

event_counts = {}

for message in consumer:
    data = message.value

    # Transform
    cleaned = {
        "user_id": data["user_id"],
        "event": data["event"].upper(),
        "amount": int(data["amount"]),
        "timestamp": data["timestamp"]
    }

    # Load
    cur.execute("""
    INSERT INTO events (user_id, event, amount, timestamp)
    VALUES (%s, %s, %s, %s)
    """, (cleaned["user_id"], cleaned["event"], cleaned["amount"], cleaned["timestamp"]))
    
    conn.commit()

    # Real-time aggregation
    event_counts[cleaned["event"]] = event_counts.get(cleaned["event"], 0) + 1

    print("Counts:", event_counts)
```


## Real-Time Analytics Queries

```python
cur.execute("""
SELECT event, COUNT(*)
FROM events
GROUP BY event
""")

print(cur.fetchall())
```


## Multiple Consumers

- Consumer 1 -> Stores data
- Consumer 2 -> Sends alerts
- Consumer 3 -> Feeds dashboard

👉 This is real production architecture


## Add Alert System

```python
if cleaned["event"] == "PURCHASE" and cleaned["amount"] > 800:
    print("🚨 High-value purchase alert!")
```


## Practice

1. Add second consumer
2. Add alert logic
3. Track top users


## Assignment

Upgrade system:

- Add:
  - multiple topics
  - batch inserts
  - error handling

Bonus:
- Integrate with Spark streaming


## How to Explain This Project

"I built a real-time analytics platform using Kafka.

- Ingested user events in real-time
- Processed events using consumers
- Stored data in PostgreSQL
- Built real-time aggregation and alert system

This system simulates production-grade streaming architecture."


## Interview Questions

1. How does Kafka handle real-time data?
2. What is offset management?
3. How do you scale Kafka consumers?
4. What are challenges in streaming pipelines?


## What You Just Built

This is elite-level:

👉 Real-time ingestion
👉 Streaming processing
👉 Database integration
👉 Live analytics
👉 Alert system

👉 This is what modern data platforms look like.


---

**Phase 6 complete.**

You now:
- Understand real-time systems
- Build streaming pipelines
- Combine Kafka + ETL + DB

👉 You are now at strong data-engineer level


## Your tasks

- [ ] Build the full pipeline end-to-end on `events-topic`.
- [ ] Add high-value purchase alert logic.
- [ ] Add a second consumer (optional but recommended).
- [ ] Capture final architecture and interview story in your project notes.
